# ML3. Часть 5. Cross-validation comparison / Сравнение схем валидации

В этом ноутбуке выполнена пятая часть задания из README:

1. применить все реализованные ранее методы валидации к датасету;
2. применить соответствующие методы из `sklearn`;
3. сравнить распределения признаков в train-части между реализацией проекта и `sklearn`;
4. сравнить схемы валидации, выбрать лучшую и объяснить выбор.

Используются методы из файлов:

* `cross_validation_methods.py`;
* `data_split_methods.py`.

Целевая переменная проекта - `price`, то есть задача является регрессионной. Для `Stratified K-Fold` непрерывный `price` предварительно преобразуется в квантильные бины `price_bin`.

## 1. Импорт библиотек и проектных методов

Импорт сделан так, чтобы ноутбук можно было запускать и из корня проекта, и из папки `src`.

In [16]:
import numpy as np
import pandas as pd
from sklearn.model_selection import (
    GroupKFold,
    KFold,
    StratifiedKFold,
    TimeSeriesSplit,
    train_test_split,
    )

from ml3_04_cv_methods_jacquieb import (
    make_group_k_fold_splits,
    make_k_fold_splits,
    make_quantile_stratification_field,
    make_stratified_k_fold_splits,
    make_time_series_splits,
    )
from ml3_03_data_split_methods_jacquieb import (
    split_by_date_train_test,
    split_by_dates_train_validation_test,
    split_random_train_test,
    split_random_train_validation_test,
    )

RANDOM_STATE = 21
DF_PATH = r'data\train_df_processed_full.csv'

## 2. Загрузка данных и подготовка target для Stratified K-Fold

Загружаем предобработанный датасет. Поле `created` приводим к `datetime`, а для стратификации регрессионного target `price` создаем `price_bin` - квантильные интервалы цены.

`price` и служебный `price_bin` исключаем из списка признаков для сравнения распределений, потому что `price` является целевой переменной.

In [17]:
df = pd.read_csv(DF_PATH)
df["created"] = pd.to_datetime(df["created"])
df["price_bin"] = make_quantile_stratification_field(df, target_field="price", n_bins=5, )

target = "price"
date_field = "created"
group_field = "bedrooms"
stratify_field = "price_bin"

feature_columns = [
    column
    for column in df.select_dtypes(include=np.number).columns
    if column not in [target, stratify_field]
    ]

print(f"Размер датасета: {df.shape}")
print(f"Количество признаков для сравнения распределений: {len(feature_columns)}")
print("Признаки:", feature_columns)

Размер датасета: (48379, 25)
Количество признаков для сравнения распределений: 22
Признаки: ['bathrooms', 'bedrooms', 'interest', 'Elevator', 'HardwoodFloors', 'CatsAllowed', 'DogsAllowed', 'Doorman', 'Dishwasher', 'NoFee', 'LaundryinBuilding', 'FitnessCenter', 'Pre-War', 'LaundryinUnit', 'RoofDeck', 'OutdoorSpace', 'DiningRoom', 'HighSpeedInternet', 'Balcony', 'SwimmingPool', 'NewConstruction', 'Terrace']


## 3. Функции сравнения распределений

Для сравнения train-частей будем считать по каждому признаку:

* среднее;
* стандартное отклонение;
* медиану;
* 25-й и 75-й перцентили.

Затем сравним статистики между нашей реализацией и реализацией `sklearn`. Чем ближе значения, тем ближе получившиеся train-выборки по распределениям признаков.

In [18]:
def feature_distribution(data: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """Возвращает базовые статистики распределения признаков."""
    return pd.DataFrame(
        {
            "mean": data[columns].mean(),
            "std": data[columns].std(),
            "median": data[columns].median(),
            "q25": data[columns].quantile(0.25),
            "q75": data[columns].quantile(0.75),
            }
        )


def compare_feature_distributions(
    custom_train: pd.DataFrame,
    sklearn_train: pd.DataFrame,
    columns: list[str],
    ) -> pd.DataFrame:
    """Сравнивает распределения признаков между custom train и sklearn train."""
    custom_stats = feature_distribution(custom_train, columns)
    sklearn_stats = feature_distribution(sklearn_train, columns)
    diff = (custom_stats - sklearn_stats).abs()

    result = diff.add_prefix("abs_diff_")
    result["max_abs_diff"] = result.max(axis=1)
    return result.sort_values("max_abs_diff", ascending=False)


def summarize_comparison(name: str, comparison: pd.DataFrame) -> dict:
    """Формирует короткую сводку по таблице различий."""
    return {
        "scheme": name,
        "mean_abs_diff_mean": comparison["abs_diff_mean"].mean(),
        "max_abs_diff_mean": comparison["abs_diff_mean"].max(),
        "mean_max_abs_diff": comparison["max_abs_diff"].mean(),
        "max_abs_diff": comparison["max_abs_diff"].max(),
        }


def fold_train_distribution_summary(
    data: pd.DataFrame,
    splits: list[tuple[np.ndarray, np.ndarray]],
    columns: list[str],
    ) -> pd.DataFrame:
    """Считает средние значения признаков в train-части каждого фолда."""
    rows = []

    for fold_number, (train_idx, test_idx) in enumerate(splits, start=1):
        train_part = data.iloc[train_idx]
        row = train_part[columns].mean().to_dict()
        row["fold"] = fold_number
        row["train_size"] = len(train_idx)
        row["test_size"] = len(test_idx)
        rows.append(row)

    return pd.DataFrame(rows).set_index("fold")


comparison_summaries = []

## 4. Random train/test split: наша реализация и sklearn

Сравниваем `split_random_train_test` с `sklearn.model_selection.train_test_split`. При одинаковых `test_size`, `random_state` и `shuffle` разбиения должны совпасть.

In [19]:
test_size = 0.20

custom_train, custom_test = split_random_train_test(
    df,
    test_size=test_size,
    random_state=RANDOM_STATE,
    )

sklearn_train, sklearn_test = train_test_split(
    df,
    test_size=test_size,
    random_state=RANDOM_STATE,
    shuffle=True,
    )

random_2_comparison = compare_feature_distributions(
    custom_train,
    sklearn_train,
    feature_columns,
    )
comparison_summaries.append(summarize_comparison("Random train/test", random_2_comparison))

print("Размеры custom:", custom_train.shape, custom_test.shape)
print("Размеры sklearn:", sklearn_train.shape, sklearn_test.shape)
print("Индексы train совпадают:", custom_train.index.equals(sklearn_train.index))
print("Индексы test совпадают:", custom_test.index.equals(sklearn_test.index))
display(random_2_comparison.head(10))

Размеры custom: (38703, 25) (9676, 25)
Размеры sklearn: (38703, 25) (9676, 25)
Индексы train совпадают: True
Индексы test совпадают: True


,abs_diff_mean,abs_diff_std,abs_diff_median,abs_diff_q25,abs_diff_q75,max_abs_diff
bathrooms,0.0,0.0,0.0,0.0,0.0,0.0
bedrooms,0.0,0.0,0.0,0.0,0.0,0.0
interest,0.0,0.0,0.0,0.0,0.0,0.0
Elevator,0.0,0.0,0.0,0.0,0.0,0.0
HardwoodFloors,0.0,0.0,0.0,0.0,0.0,0.0
CatsAllowed,0.0,0.0,0.0,0.0,0.0,0.0
DogsAllowed,0.0,0.0,0.0,0.0,0.0,0.0
Doorman,0.0,0.0,0.0,0.0,0.0,0.0
Dishwasher,0.0,0.0,0.0,0.0,0.0,0.0
NoFee,0.0,0.0,0.0,0.0,0.0,0.0


## 5. Random train/validation/test split: наша реализация и sklearn

Для разбиения на три части используем два последовательных `train_test_split`. В нашей функции используется та же логика: сначала отделяется `test`, затем из оставшейся части отделяется `validation`.

In [20]:
validation_size = 0.20
test_size = 0.20

custom_train, custom_validation, custom_test = split_random_train_validation_test(
    df,
    validation_size=validation_size,
    test_size=test_size,
    random_state=RANDOM_STATE,
    )

sklearn_train_validation, sklearn_test = train_test_split(
    df,
    test_size=test_size,
    random_state=RANDOM_STATE,
    shuffle=True,
    )
adjusted_validation_size = validation_size / (1 - test_size)
sklearn_train, sklearn_validation = train_test_split(
    sklearn_train_validation,
    test_size=adjusted_validation_size,
    random_state=RANDOM_STATE,
    shuffle=True,
    )

random_3_comparison = compare_feature_distributions(
    custom_train,
    sklearn_train,
    feature_columns,
    )
comparison_summaries.append(summarize_comparison("Random train/validation/test", random_3_comparison))

print("Размеры custom:", custom_train.shape, custom_validation.shape, custom_test.shape)
print("Размеры sklearn:", sklearn_train.shape, sklearn_validation.shape, sklearn_test.shape)
print("Индексы train совпадают:", custom_train.index.equals(sklearn_train.index))
print("Индексы validation совпадают:", custom_validation.index.equals(sklearn_validation.index))
print("Индексы test совпадают:", custom_test.index.equals(sklearn_test.index))
display(random_3_comparison.head(10))

Размеры custom: (29027, 25) (9676, 25) (9676, 25)
Размеры sklearn: (29027, 25) (9676, 25) (9676, 25)
Индексы train совпадают: True
Индексы validation совпадают: True
Индексы test совпадают: True


,abs_diff_mean,abs_diff_std,abs_diff_median,abs_diff_q25,abs_diff_q75,max_abs_diff
bathrooms,0.0,0.0,0.0,0.0,0.0,0.0
bedrooms,0.0,0.0,0.0,0.0,0.0,0.0
interest,0.0,0.0,0.0,0.0,0.0,0.0
Elevator,0.0,0.0,0.0,0.0,0.0,0.0
HardwoodFloors,0.0,0.0,0.0,0.0,0.0,0.0
CatsAllowed,0.0,0.0,0.0,0.0,0.0,0.0
DogsAllowed,0.0,0.0,0.0,0.0,0.0,0.0
Doorman,0.0,0.0,0.0,0.0,0.0,0.0
Dishwasher,0.0,0.0,0.0,0.0,0.0,0.0
NoFee,0.0,0.0,0.0,0.0,0.0,0.0


## 6. Date split: train/test

В `sklearn` нет прямого аналога разбиения по фиксированной дате. Поэтому в качестве эталонной реализации используем явную boolean-mask логику на pandas. Это тот же принцип: все наблюдения до `date_split` идут в train, остальные - в test.

In [21]:
date_split = "2016-06-01"

custom_train, custom_test = split_by_date_train_test(
    df,
    date_split=date_split,
    date_field=date_field,
    )

date_split_ts = pd.to_datetime(date_split)
sklearn_like_train = df[df[date_field] < date_split_ts].copy()
sklearn_like_test = df[df[date_field] >= date_split_ts].copy()

date_2_comparison = compare_feature_distributions(
    custom_train,
    sklearn_like_train,
    feature_columns,
    )
comparison_summaries.append(summarize_comparison("Date train/test", date_2_comparison))

print("Размеры custom:", custom_train.shape, custom_test.shape)
print("Размеры reference:", sklearn_like_train.shape, sklearn_like_test.shape)
print("Индексы train совпадают:", custom_train.index.equals(sklearn_like_train.index))
print("Индексы test совпадают:", custom_test.index.equals(sklearn_like_test.index))
print("train date range:", custom_train[date_field].min(), "-", custom_train[date_field].max())
print("test date range:", custom_test[date_field].min(), "-", custom_test[date_field].max())
display(date_2_comparison.head(10))

Размеры custom: (31576, 25) (16803, 25)
Размеры reference: (31576, 25) (16803, 25)
Индексы train совпадают: True
Индексы test совпадают: True
train date range: 2016-04-01 22:12:41 - 2016-05-31 23:10:48
test date range: 2016-06-01 01:10:37 - 2016-06-29 21:41:47


,abs_diff_mean,abs_diff_std,abs_diff_median,abs_diff_q25,abs_diff_q75,max_abs_diff
bathrooms,0.0,0.0,0.0,0.0,0.0,0.0
bedrooms,0.0,0.0,0.0,0.0,0.0,0.0
interest,0.0,0.0,0.0,0.0,0.0,0.0
Elevator,0.0,0.0,0.0,0.0,0.0,0.0
HardwoodFloors,0.0,0.0,0.0,0.0,0.0,0.0
CatsAllowed,0.0,0.0,0.0,0.0,0.0,0.0
DogsAllowed,0.0,0.0,0.0,0.0,0.0,0.0
Doorman,0.0,0.0,0.0,0.0,0.0,0.0
Dishwasher,0.0,0.0,0.0,0.0,0.0,0.0
NoFee,0.0,0.0,0.0,0.0,0.0,0.0


## 7. Date split: train/validation/test

Аналогично предыдущему пункту: прямого метода в `sklearn` для `validation_date` и `test_date` нет, поэтому сравниваем с эквивалентной pandas-реализацией.

In [22]:
validation_date = "2016-05-01"
test_date = "2016-06-01"

custom_train, custom_validation, custom_test = split_by_dates_train_validation_test(
    df,
    validation_date=validation_date,
    test_date=test_date,
    date_field=date_field,
    )

validation_date_ts = pd.to_datetime(validation_date)
test_date_ts = pd.to_datetime(test_date)
sklearn_like_train = df[df[date_field] < validation_date_ts].copy()
sklearn_like_validation = df[
    (df[date_field] >= validation_date_ts)
    & (df[date_field] < test_date_ts)
    ].copy()
sklearn_like_test = df[df[date_field] >= test_date_ts].copy()

date_3_comparison = compare_feature_distributions(
    custom_train,
    sklearn_like_train,
    feature_columns,
    )
comparison_summaries.append(summarize_comparison("Date train/validation/test", date_3_comparison))

print("Размеры custom:", custom_train.shape, custom_validation.shape, custom_test.shape)
print("Размеры reference:", sklearn_like_train.shape, sklearn_like_validation.shape, sklearn_like_test.shape)
print("Индексы train совпадают:", custom_train.index.equals(sklearn_like_train.index))
print("Индексы validation совпадают:", custom_validation.index.equals(sklearn_like_validation.index))
print("Индексы test совпадают:", custom_test.index.equals(sklearn_like_test.index))
display(date_3_comparison.head(10))

Размеры custom: (16102, 25) (15474, 25) (16803, 25)
Размеры reference: (16102, 25) (15474, 25) (16803, 25)
Индексы train совпадают: True
Индексы validation совпадают: True
Индексы test совпадают: True


,abs_diff_mean,abs_diff_std,abs_diff_median,abs_diff_q25,abs_diff_q75,max_abs_diff
bathrooms,0.0,0.0,0.0,0.0,0.0,0.0
bedrooms,0.0,0.0,0.0,0.0,0.0,0.0
interest,0.0,0.0,0.0,0.0,0.0,0.0
Elevator,0.0,0.0,0.0,0.0,0.0,0.0
HardwoodFloors,0.0,0.0,0.0,0.0,0.0,0.0
CatsAllowed,0.0,0.0,0.0,0.0,0.0,0.0
DogsAllowed,0.0,0.0,0.0,0.0,0.0,0.0
Doorman,0.0,0.0,0.0,0.0,0.0,0.0
Dishwasher,0.0,0.0,0.0,0.0,0.0,0.0
NoFee,0.0,0.0,0.0,0.0,0.0,0.0


## 8. K-Fold: наша реализация и sklearn

Сравниваем нашу `make_k_fold_splits` с `sklearn.model_selection.KFold`. При одинаковом `k`, `shuffle=True` и `random_state=21` размеры фолдов должны совпадать, но порядок конкретных индексов может отличаться, потому что наша реализация использует `np.random.default_rng`, а `sklearn` - собственную логику случайного перемешивания.

Поэтому сравниваем не полное совпадение индексов, а распределения признаков в train-частях.

In [23]:
k = 5

custom_splits = make_k_fold_splits(df, k=k, shuffle=True, random_state=RANDOM_STATE)
sklearn_splits = list(
    KFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE).split(df)
    )

custom_train_summary = fold_train_distribution_summary(df, custom_splits, feature_columns)
sklearn_train_summary = fold_train_distribution_summary(df, sklearn_splits, feature_columns)

kfold_comparison = (custom_train_summary[feature_columns] - sklearn_train_summary[feature_columns]).abs().T
kfold_comparison["max_abs_diff"] = kfold_comparison.max(axis=1)
kfold_comparison = kfold_comparison.sort_values("max_abs_diff", ascending=False)
comparison_summaries.append(
    {
        "scheme": "K-Fold",
        "mean_abs_diff_mean": kfold_comparison.drop(columns="max_abs_diff").mean(axis=1).mean(),
        "max_abs_diff_mean": kfold_comparison.drop(columns="max_abs_diff").mean(axis=1).max(),
        "mean_max_abs_diff": kfold_comparison["max_abs_diff"].mean(),
        "max_abs_diff": kfold_comparison["max_abs_diff"].max(),
        }
    )

print("Размеры фолдов custom:")
print([(len(train_idx), len(test_idx)) for train_idx, test_idx in custom_splits])
print("Размеры фолдов sklearn:")
print([(len(train_idx), len(test_idx)) for train_idx, test_idx in sklearn_splits])
display(kfold_comparison.head(10))

Размеры фолдов custom:
[(38703, 9676), (38703, 9676), (38703, 9676), (38703, 9676), (38704, 9675)]
Размеры фолдов sklearn:
[(38703, 9676), (38703, 9676), (38703, 9676), (38703, 9676), (38704, 9675)]


fold,1,2,3,4,5,max_abs_diff
bedrooms,0.007803,0.000904,0.003927,0.002403,0.002377,0.007803
NoFee,0.002997,0.001344,0.002274,0.003488,0.000439,0.003488
Dishwasher,0.000052,0.001395,0.003204,0.000129,0.001989,0.003204
HardwoodFloors,0.000853,0.000982,0.003023,0.002300,0.000594,0.003023
LaundryinUnit,0.002480,0.001576,0.001214,0.000026,0.000336,0.002480
DogsAllowed,0.001834,0.000181,0.002067,0.000413,0.000827,0.002067
interest,0.001783,0.000052,0.000594,0.001964,0.000465,0.001964
CatsAllowed,0.001731,0.000620,0.001964,0.000827,0.001679,0.001964
Elevator,0.000026,0.000517,0.001705,0.000233,0.000930,0.001705
bathrooms,0.001589,0.000698,0.001008,0.001563,0.001447,0.001589


## 9. Grouped K-Fold: наша реализация и sklearn

Для демонстрации группового разбиения используем `bedrooms` как `group_field`.

In [28]:
k = 5

custom_splits = make_group_k_fold_splits(df, k=k, group_field=group_field)
sklearn_splits = list(GroupKFold(n_splits=k).split(df, groups=df[group_field]))

custom_train_summary = fold_train_distribution_summary(df, custom_splits, feature_columns)
sklearn_train_summary = fold_train_distribution_summary(df, sklearn_splits, feature_columns)

group_comparison = (custom_train_summary[feature_columns] - sklearn_train_summary[feature_columns]).abs().T
group_comparison["max_abs_diff"] = group_comparison.max(axis=1)
group_comparison = group_comparison.sort_values("max_abs_diff", ascending=False)
comparison_summaries.append(
    {
        "scheme": "Grouped K-Fold",
        "mean_abs_diff_mean": group_comparison.drop(columns="max_abs_diff").mean(axis=1).mean(),
        "max_abs_diff_mean": group_comparison.drop(columns="max_abs_diff").mean(axis=1).max(),
        "mean_max_abs_diff": group_comparison["max_abs_diff"].mean(),
        "max_abs_diff": group_comparison["max_abs_diff"].max(),
        }
    )

print("Проверка отсутствия пересечений групп в custom:")
for fold_number, (train_idx, test_idx) in enumerate(custom_splits, start=1):
    train_groups = set(df.iloc[train_idx][group_field])
    test_groups = set(df.iloc[test_idx][group_field])
    print(f"Fold {fold_number}: пересечение групп = {sorted(train_groups & test_groups)}")

display(group_comparison.head(10))

Проверка отсутствия пересечений групп в custom:
Fold 1: пересечение групп = []
Fold 2: пересечение групп = []
Fold 3: пересечение групп = []
Fold 4: пересечение групп = []
Fold 5: пересечение групп = []


fold,1,2,3,4,5,max_abs_diff
bathrooms,0.0,0.0,0.0,0.0,0.0,0.0
bedrooms,0.0,0.0,0.0,0.0,0.0,0.0
interest,0.0,0.0,0.0,0.0,0.0,0.0
Elevator,0.0,0.0,0.0,0.0,0.0,0.0
HardwoodFloors,0.0,0.0,0.0,0.0,0.0,0.0
CatsAllowed,0.0,0.0,0.0,0.0,0.0,0.0
DogsAllowed,0.0,0.0,0.0,0.0,0.0,0.0
Doorman,0.0,0.0,0.0,0.0,0.0,0.0
Dishwasher,0.0,0.0,0.0,0.0,0.0,0.0
NoFee,0.0,0.0,0.0,0.0,0.0,0.0


## 10. Stratified K-Fold: наша реализация и sklearn

`price` - непрерывная целевая переменная, поэтому для стратификации используем `price_bin`. Это позволяет сохранить похожее распределение ценовых диапазонов в каждом фолде.

In [25]:
k = 5

custom_splits = make_stratified_k_fold_splits(
    df,
    k=k,
    stratify_field=stratify_field,
    shuffle=True,
    random_state=RANDOM_STATE,
    )
sklearn_splits = list(
    StratifiedKFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE).split(
        df,
        df[stratify_field],
        )
    )

custom_train_summary = fold_train_distribution_summary(df, custom_splits, feature_columns)
sklearn_train_summary = fold_train_distribution_summary(df, sklearn_splits, feature_columns)

stratified_comparison = (custom_train_summary[feature_columns] - sklearn_train_summary[feature_columns]).abs().T
stratified_comparison["max_abs_diff"] = stratified_comparison.max(axis=1)
stratified_comparison = stratified_comparison.sort_values("max_abs_diff", ascending=False)
comparison_summaries.append(
    {
        "scheme": "Stratified K-Fold by price_bin",
        "mean_abs_diff_mean": stratified_comparison.drop(columns="max_abs_diff").mean(axis=1).mean(),
        "max_abs_diff_mean": stratified_comparison.drop(columns="max_abs_diff").mean(axis=1).max(),
        "mean_max_abs_diff": stratified_comparison["max_abs_diff"].mean(),
        "max_abs_diff": stratified_comparison["max_abs_diff"].max(),
        }
    )

print("Распределение price_bin в полном датасете:")
display(df[stratify_field].value_counts(normalize=True).sort_index())

print("Распределение price_bin в test-фолдах custom:")
for fold_number, (_, test_idx) in enumerate(custom_splits, start=1):
    print(f"Fold {fold_number}")
    display(df.iloc[test_idx][stratify_field].value_counts(normalize=True).sort_index())

display(stratified_comparison.head(10))

Распределение price_bin в полном датасете:


price_bin
0    0.200314
1    0.199715
2    0.205709
3    0.195581
4    0.198681
Name: proportion, dtype: float64

Распределение price_bin в test-фолдах custom:
Fold 1


price_bin
0    0.200331
1    0.199711
2    0.205703
3    0.195578
4    0.198678
Name: proportion, dtype: float64

Fold 2


...

Fold 4


price_bin
0    0.200331
1    0.199711
2    0.205706
3    0.195576
4    0.198677
Name: proportion, dtype: float64

Fold 5


price_bin
0    0.200331
1    0.199711
2    0.205706
3    0.195576
4    0.198677
Name: proportion, dtype: float64

fold,1,2,3,4,5,max_abs_diff
bedrooms,0.006444,0.006616,0.004937,0.001061,0.005825,0.006616
CatsAllowed,0.000864,0.000852,0.001136,0.002430,0.003009,0.003009
interest,0.002994,0.000640,0.000446,0.001970,0.000061,0.002994
OutdoorSpace,0.000783,0.002423,0.000135,0.001648,0.000126,0.002423
HardwoodFloors,0.002363,0.001988,0.000671,0.002118,0.001563,0.002363
DogsAllowed,0.001017,0.000979,0.001367,0.001579,0.002208,0.002208
Pre-War,0.001203,0.002058,0.001411,0.000611,0.001167,0.002058
FitnessCenter,0.001959,0.001642,0.000221,0.001926,0.001454,0.001959
SwimmingPool,0.000082,0.000359,0.001909,0.000778,0.000854,0.001909
Elevator,0.001875,0.000260,0.000076,0.001009,0.001202,0.001875


## 11. Time series split: наша реализация и sklearn

Для честного сравнения сначала сортируем данные по `created`, а затем применяем и нашу реализацию, и `sklearn.model_selection.TimeSeriesSplit` к отсортированному DataFrame.

В нашей реализации параметр `k` означает количество временных блоков, поэтому получается `k - 1` разбиений. У `TimeSeriesSplit` напрямую задается `n_splits`, поэтому берем `n_splits = k - 1`.

In [26]:
k = 5
df_sorted = df.sort_values(date_field).reset_index(drop=True)

custom_splits = make_time_series_splits(df_sorted, k=k, date_field=date_field)
sklearn_splits = list(TimeSeriesSplit(n_splits=k - 1).split(df_sorted))

custom_train_summary = fold_train_distribution_summary(df_sorted, custom_splits, feature_columns)
sklearn_train_summary = fold_train_distribution_summary(df_sorted, sklearn_splits, feature_columns)

time_comparison = (custom_train_summary[feature_columns] - sklearn_train_summary[feature_columns]).abs().T
time_comparison["max_abs_diff"] = time_comparison.max(axis=1)
time_comparison = time_comparison.sort_values("max_abs_diff", ascending=False)
comparison_summaries.append(
    {
        "scheme": "Time series split",
        "mean_abs_diff_mean": time_comparison.drop(columns="max_abs_diff").mean(axis=1).mean(),
        "max_abs_diff_mean": time_comparison.drop(columns="max_abs_diff").mean(axis=1).max(),
        "mean_max_abs_diff": time_comparison["max_abs_diff"].mean(),
        "max_abs_diff": time_comparison["max_abs_diff"].max(),
        }
    )

print("Диапазоны дат в custom split:")
for fold_number, (train_idx, test_idx) in enumerate(custom_splits, start=1):
    train_dates = df_sorted.iloc[train_idx][date_field]
    test_dates = df_sorted.iloc[test_idx][date_field]
    print(
        f"Fold {fold_number}: "
        f"train {train_dates.min()} - {train_dates.max()}, "
        f"test {test_dates.min()} - {test_dates.max()}"
        )

print("Размеры фолдов sklearn TimeSeriesSplit:")
print([(len(train_idx), len(test_idx)) for train_idx, test_idx in sklearn_splits])
display(time_comparison.head(10))

Диапазоны дат в custom split:
Fold 1: train 2016-04-01 22:12:41 - 2016-04-19 03:57:46, test 2016-04-19 03:59:13 - 2016-05-07 02:58:20
Fold 2: train 2016-04-01 22:12:41 - 2016-05-07 02:58:20, test 2016-05-07 02:58:23 - 2016-05-24 16:40:27
Fold 3: train 2016-04-01 22:12:41 - 2016-05-24 16:40:27, test 2016-05-24 16:44:06 - 2016-06-12 08:07:50
Fold 4: train 2016-04-01 22:12:41 - 2016-06-12 08:07:50, test 2016-06-12 08:09:25 - 2016-06-29 21:41:47
Размеры фолдов sklearn TimeSeriesSplit:
[(9679, 9675), (19354, 9675), (29029, 9675), (38704, 9675)]


fold,1,2,3,4,max_abs_diff
bedrooms,0.000260,0.000053,0.000018,0.0,0.000260
DogsAllowed,0.000171,0.000046,0.000019,0.0,0.000171
CatsAllowed,0.000162,0.000049,0.000018,0.0,0.000162
HardwoodFloors,0.000150,0.000053,0.000017,0.0,0.000150
bathrooms,0.000149,0.000020,0.000007,0.0,0.000149
Dishwasher,0.000128,0.000060,0.000014,0.0,0.000128
LaundryinBuilding,0.000103,0.000017,0.000012,0.0,0.000103
Elevator,0.000058,0.000049,0.000018,0.0,0.000058
LaundryinUnit,0.000054,0.000033,0.000006,0.0,0.000054
Pre-War,0.000054,0.000019,0.000006,0.0,0.000054


## 12. Сводная таблица сравнения custom и sklearn/reference

В таблице ниже показано, насколько сильно отличаются распределения признаков в train-частях между нашей реализацией и соответствующей реализацией из `sklearn` или эквивалентной reference-логикой.

In [27]:
comparison_summary = pd.DataFrame(comparison_summaries).set_index("scheme")
display(comparison_summary.sort_values("mean_max_abs_diff"))

,mean_abs_diff_mean,max_abs_diff_mean,mean_max_abs_diff,max_abs_diff
scheme,,,,
Random train/test,0.000000,0.000000,0.000000,0.000000
Random train/validation/test,0.000000,0.000000,0.000000,0.000000
Date train/test,0.000000,0.000000,0.000000,0.000000
Date train/validation/test,0.000000,0.000000,0.000000,0.000000
Grouped K-Fold,0.000000,0.000000,0.000000,0.000000
Time series split,0.000027,0.000083,0.000074,0.000260
K-Fold,0.000950,0.003483,0.001814,0.007803
Stratified K-Fold by price_bin,0.001139,0.004977,0.001972,0.006616


## 13. Сравнение всех схем валидации

Коротко о схемах:

* **Random train/test** - простая базовая схема. Подходит, если объекты можно считать независимыми и нет временной зависимости.
* **Random train/validation/test** - удобна для подбора параметров модели, потому что есть отдельная validation-часть.
* **Date split** - лучше имитирует production-сценарий, если модель должна работать на будущих данных.
* **K-Fold** - дает более устойчивую оценку качества, чем один train/test split, но игнорирует время и группы.
* **Grouped K-Fold** - нужен, если есть настоящие группы и нельзя допускать пересечения группы между train и test. В текущем датасете мы использовали `bedrooms` только как технический пример, поэтому делать финальный выбор в пользу Grouped K-Fold по этому полю не стоит.
* **Stratified K-Fold by price_bin** - полезен для регрессии, если нужно сохранить похожее распределение целевой переменной `price` в каждом фолде. Для этого `price` разбивается на квантильные интервалы.
* **Time series split** - наиболее честная схема, если важен временной порядок: модель обучается на прошлом и проверяется на будущем.

## 14. Выбор лучшей схемы

Для этого датасета лучшей основной схемой я выбираю **Time series split** по полю `created`.

Причины:

1. В данных есть дата создания объявления `created`, значит наблюдения имеют естественный временной порядок.
2. В реальном применении модель, скорее всего, будет обучаться на уже известных объявлениях и применяться к новым объявлениям из будущего.
3. `Time series split` снижает риск утечки будущей информации в обучение.
4. Схема проверяет модель в нескольких последовательных временных окнах, а не только на одном фиксированном test-периоде.

Если бы мы были уверены, что временная зависимость не важна и данные можно считать независимыми, хорошей альтернативой был бы **Stratified K-Fold by price_bin**: он сохраняет распределение целевой переменной `price` по фолдам и дает устойчивую оценку качества. Но для production-сценария с будущими объявлениями более честным выбором остается **Time series split**.